In [24]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ==========================================
# Device Configuration
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device : {device}")

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

# ==========================================
# Dataset
# ==========================================

class ColorizationDataset(Dataset):

    def __init__(self, image_folder):
        self.image_folder = image_folder

        self.image_list = [
            file for file in os.listdir(image_folder)
            if file.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):

        image_path = os.path.join(
            self.image_folder,
            self.image_list[idx]
        )

        image = cv2.imread(image_path)

        image = cv2.resize(image, (256, 256))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

        # Normalize

        L = lab[:, :, 0].astype(np.float32) / 255.0

        A = (lab[:, :, 1].astype(np.float32) - 128.0) / 128.0

        B = (lab[:, :, 2].astype(np.float32) - 128.0) / 128.0

        L = np.expand_dims(L, axis=0)

        AB = np.stack((A, B), axis=0)

        return (
            torch.tensor(L, dtype=torch.float32),
            torch.tensor(AB, dtype=torch.float32)
        )

# ==========================================
# Mini U-Net
# ==========================================

class MiniUNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(

            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)

        )

        # Bottleneck
        self.bottleneck = nn.Sequential(

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)

        )

        # Decoder
        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(
                128,
                64,
                kernel_size=2,
                stride=2
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            )

        )

    def forward(self, x):

        x = self.encoder(x)

        x = self.bottleneck(x)

        x = self.decoder(x)

        return x

# ==========================================
# Main
# ==========================================

def main():

    IMAGE_FOLDER = "/kaggle/input/datasets/aayush9753/image-colorization-dataset/data/train_color"

    dataset = ColorizationDataset(IMAGE_FOLDER)

    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = MiniUNet().to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    epochs = 10

    print("\nTraining Started...\n")

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for L, AB in loader:

            L = L.to(device, non_blocking=True)
            AB = AB.to(device, non_blocking=True)

            optimizer.zero_grad()

            prediction = model(L)

            loss = criterion(prediction, AB)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(loader)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f}"
        )

    torch.save(
        model.state_dict(),
        "saved_model.pth"
    )

    print("\nTraining Completed!")
    print("Model saved as saved_model.pth")


if __name__ == "__main__":
    main()

Using Device : cuda
GPU : Tesla T4

Training Started...

Epoch [1/10] Loss: 0.0126
Epoch [2/10] Loss: 0.0124
Epoch [3/10] Loss: 0.0123
Epoch [4/10] Loss: 0.0122
Epoch [5/10] Loss: 0.0121
Epoch [6/10] Loss: 0.0120
Epoch [7/10] Loss: 0.0120
Epoch [8/10] Loss: 0.0119
Epoch [9/10] Loss: 0.0119
Epoch [10/10] Loss: 0.0119

Training Completed!
Model saved as saved_model.pth
